# Semana 16 — API para un agente asíncrono

**Objetivo:** exponer ResearchAgent como API REST con el patrón fire-and-poll:
`POST /research` lanza la investigación en background y devuelve un `job_id`;
el cliente hace polling con `GET /jobs/{job_id}` hasta que el status sea `completed`.

**Lo que construyes** en `proyectos/research-agent/`:
- `models.py` — Pydantic models para request/response
- `jobs.py` — almacén de jobs + máquina de estados
- `api.py` — FastAPI con 6 endpoints
- `test_api.py` — tests con TestClient (incluyendo polling simulado)

**Prerrequisitos:** semana 13 (FastAPI + Docker), semana 15 (ResearchAgent).

**Coste estimado:** $0.02–0.10 si ejecutas tests que llamen al agente real.

## Celda 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn[standard]", "httpx", "pytest",
                "python-dotenv"])

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Define OPENAI_API_KEY en tu .env"

PROJECT_DIR = Path("../proyectos/research-agent").resolve()
assert PROJECT_DIR.exists(), "Ejecuta primero el notebook de la semana 15"
print(f"✓ Proyecto: {PROJECT_DIR}")

## Celda 1 — Modelos Pydantic (`models.py`)

In [ ]:
MODELS_PY = '''
"""Modelos de request y response para la ResearchAgent API."""
from enum import Enum
from datetime import datetime
from pydantic import BaseModel, Field


class EstadoJob(str, Enum):
    PENDING   = "pending"
    RUNNING   = "running"
    COMPLETED = "completed"
    FAILED    = "failed"


# ── Requests ──────────────────────────────────────────────────────────────────

class ResearchRequest(BaseModel):
    tema: str = Field(
        min_length=10, max_length=300,
        description="Tema a investigar",
        examples=["Aplicaciones de IA generativa en acuicultura en Europa"],
    )


# ── Responses ─────────────────────────────────────────────────────────────────

class JobCreatedResponse(BaseModel):
    job_id: str
    status: EstadoJob
    mensaje: str


class JobStatusResponse(BaseModel):
    job_id:        str
    tema:          str
    status:        EstadoJob
    progreso:      str
    creado_en:     datetime
    completado_en: datetime | None = None
    n_fuentes:     int             = 0
    duracion_s:    float           = 0.0
    error:         str             = ""


class ReportResponse(BaseModel):
    job_id:    str
    tema:      str
    informe:   str
    n_fuentes: int
    duracion_s: float


class JobListResponse(BaseModel):
    total: int
    jobs:  list[JobStatusResponse]


class HealthResponse(BaseModel):
    status:      str
    jobs_activos: int


class ErrorResponse(BaseModel):
    detail: str
'''

(PROJECT_DIR / "models.py").write_text(MODELS_PY.strip())
print("✓ models.py generado")

# Verifica que Pydantic valida correctamente
import sys
sys.path.insert(0, str(PROJECT_DIR))

from models import ResearchRequest

# OK
r = ResearchRequest(tema="Inteligencia artificial en la pesca gallega")
print(f"✓ Validación OK: {r.tema}")

# Debe fallar
try:
    ResearchRequest(tema="corto")
    print("✗ Debería haber fallado")
except Exception as e:
    print(f"✓ Validación min_length correcta: {type(e).__name__}")

## Celda 2 — Almacén de jobs (`jobs.py`)

Gestiona el ciclo de vida de los jobs: crear, actualizar estado, consultar, eliminar.

In [ ]:
JOBS_PY = '''
"""Almacén en memoria para el ciclo de vida de los jobs de investigación."""
import threading
from dataclasses import dataclass, field
from datetime import datetime
from models import EstadoJob


@dataclass
class Job:
    id:            str
    tema:          str
    status:        EstadoJob      = EstadoJob.PENDING
    progreso:      str            = "En cola"
    informe:       str            = ""
    error:         str            = ""
    creado_en:     datetime       = field(default_factory=datetime.utcnow)
    completado_en: datetime|None  = None
    n_fuentes:     int            = 0
    duracion_s:    float          = 0.0


class JobStore:
    """Almacén thread-safe de jobs en memoria."""

    def __init__(self):
        self._jobs: dict[str, Job] = {}
        self._lock = threading.Lock()

    def crear(self, job_id: str, tema: str) -> Job:
        job = Job(id=job_id, tema=tema)
        with self._lock:
            self._jobs[job_id] = job
        return job

    def get(self, job_id: str) -> Job | None:
        with self._lock:
            return self._jobs.get(job_id)

    def listar(self) -> list[Job]:
        with self._lock:
            return sorted(self._jobs.values(),
                          key=lambda j: j.creado_en, reverse=True)

    def eliminar(self, job_id: str) -> bool:
        with self._lock:
            return self._jobs.pop(job_id, None) is not None

    def actualizar(self, job_id: str, **kwargs):
        """Actualiza campos del job de forma thread-safe."""
        with self._lock:
            job = self._jobs.get(job_id)
            if job:
                for k, v in kwargs.items():
                    setattr(job, k, v)

    def n_activos(self) -> int:
        """Número de jobs en estado pending o running."""
        with self._lock:
            return sum(
                1 for j in self._jobs.values()
                if j.status in (EstadoJob.PENDING, EstadoJob.RUNNING)
            )


# Instancia global — compartida por todos los endpoints
store = JobStore()
'''

(PROJECT_DIR / "jobs.py").write_text(JOBS_PY.strip())
print("✓ jobs.py generado")

# Prueba unitaria del store
import importlib, jobs
importlib.reload(jobs)
from jobs import JobStore
from models import EstadoJob

s = JobStore()
j = s.crear("abc123", "Tema de prueba con suficientes caracteres")
assert j.status == EstadoJob.PENDING

s.actualizar("abc123", status=EstadoJob.RUNNING, progreso="1/3 subtareas")
assert s.get("abc123").status == EstadoJob.RUNNING
assert s.n_activos() == 1

s.actualizar("abc123", status=EstadoJob.COMPLETED)
assert s.n_activos() == 0

assert s.eliminar("abc123")
assert s.get("abc123") is None

print("✓ JobStore: todos los tests pasaron")

## Celda 3 — FastAPI con 6 endpoints (`api.py`)

In [ ]:
API_PY = '''
"""ResearchAgent API — patrón fire-and-poll para investigación asíncrona."""
import asyncio
import uuid
import os
from contextlib import asynccontextmanager
from datetime import datetime

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from models import (
    EstadoJob, ResearchRequest,
    JobCreatedResponse, JobStatusResponse, ReportResponse,
    JobListResponse, HealthResponse,
)
from jobs import store
from agent.agent import ResearchAgent

INFORMES_DIR = os.getenv("INFORMES_DIR", "./informes")


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Arranque
    import pathlib
    pathlib.Path(INFORMES_DIR).mkdir(parents=True, exist_ok=True)
    yield
    # Shutdown: nada que cerrar (el store vive en memoria)


app = FastAPI(
    title="ResearchAgent API",
    description="Agente de investigación autónomo — patrón fire-and-poll",
    version="1.0.0",
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# ── Background task ───────────────────────────────────────────────────────────

async def _ejecutar_investigacion(job_id: str, tema: str):
    """Corre ResearchAgent en un thread separado para no bloquear el event loop."""
    store.actualizar(job_id, status=EstadoJob.RUNNING, progreso="Planificando...")
    try:
        agente   = ResearchAgent(informes_dir=INFORMES_DIR, verbose=False)
        resultado = await asyncio.to_thread(agente.investigar, tema)

        store.actualizar(
            job_id,
            status        = EstadoJob.COMPLETED,
            progreso      = "Completado",
            informe       = resultado.informe,
            n_fuentes     = resultado.n_fuentes,
            duracion_s    = resultado.duracion_s,
            completado_en = datetime.utcnow(),
        )
    except Exception as exc:
        store.actualizar(
            job_id,
            status   = EstadoJob.FAILED,
            progreso = "Error",
            error    = str(exc),
        )


# ── Endpoints ─────────────────────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse)
async def health():
    return HealthResponse(status="ok", jobs_activos=store.n_activos())


@app.post("/research", response_model=JobCreatedResponse, status_code=202)
async def crear_investigacion(req: ResearchRequest) -> JobCreatedResponse:
    """Inicia una investigación en background. Devuelve job_id inmediatamente."""
    job_id = str(uuid.uuid4())[:8]
    store.crear(job_id, req.tema)
    asyncio.create_task(_ejecutar_investigacion(job_id, req.tema))
    return JobCreatedResponse(
        job_id  = job_id,
        status  = EstadoJob.PENDING,
        mensaje = f"Investigación iniciada. Consulta el estado en GET /jobs/{job_id}",
    )


@app.get("/jobs/{job_id}", response_model=JobStatusResponse)
async def estado_job(job_id: str) -> JobStatusResponse:
    """Devuelve el estado y progreso del job."""
    job = store.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail=f"Job {job_id!r} no encontrado")
    return JobStatusResponse(
        job_id        = job.id,
        tema          = job.tema,
        status        = job.status,
        progreso      = job.progreso,
        creado_en     = job.creado_en,
        completado_en = job.completado_en,
        n_fuentes     = job.n_fuentes,
        duracion_s    = job.duracion_s,
        error         = job.error,
    )


@app.get("/jobs/{job_id}/report", response_model=ReportResponse)
async def obtener_informe(job_id: str) -> ReportResponse:
    """Devuelve el informe completo. Solo disponible cuando status=completed."""
    job = store.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail=f"Job {job_id!r} no encontrado")
    if job.status != EstadoJob.COMPLETED:
        raise HTTPException(
            status_code=409,
            detail=f"El informe aún no está disponible (status: {job.status})",
        )
    return ReportResponse(
        job_id     = job.id,
        tema       = job.tema,
        informe    = job.informe,
        n_fuentes  = job.n_fuentes,
        duracion_s = job.duracion_s,
    )


@app.get("/jobs", response_model=JobListResponse)
async def listar_jobs() -> JobListResponse:
    """Lista todos los jobs ordenados por fecha de creación (más reciente primero)."""
    jobs = store.listar()
    return JobListResponse(
        total=len(jobs),
        jobs=[
            JobStatusResponse(
                job_id=j.id, tema=j.tema, status=j.status,
                progreso=j.progreso, creado_en=j.creado_en,
                completado_en=j.completado_en, n_fuentes=j.n_fuentes,
                duracion_s=j.duracion_s, error=j.error,
            )
            for j in jobs
        ],
    )


@app.delete("/jobs/{job_id}", status_code=204)
async def eliminar_job(job_id: str):
    """Elimina un job del almacén."""
    if not store.eliminar(job_id):
        raise HTTPException(status_code=404, detail=f"Job {job_id!r} no encontrado")
'''

(PROJECT_DIR / "api.py").write_text(API_PY.strip())
print("✓ api.py generado")
print()
print("Endpoints:")
for linea in API_PY.split("\n"):
    if linea.strip().startswith("@app."):
        print(" ", linea.strip())

## Celda 4 — Tests (`test_api.py`)

Los tests usan `TestClient` de FastAPI. El reto de esta API es que el agente tarda 30-120s,
así que los tests rápidos **mockean** `ResearchAgent.investigar()` y hay un test lento
separado que llama al agente real.

In [ ]:
TEST_API_PY = '''
"""
Tests para la ResearchAgent API.

Tests rápidos (mock): pytest test_api.py -v -m "not lento"
Tests completos:      pytest test_api.py -v  (requiere OPENAI_API_KEY y ~2min)
"""
import asyncio
import time
import pytest
from unittest.mock import patch, MagicMock
from fastapi.testclient import TestClient

from api import app
from jobs import store
from models import EstadoJob


# ── Fixture: mock del agente ──────────────────────────────────────────────────

def _mock_resultado():
    """Resultado simulado de ResearchAgent.investigar()."""
    r = MagicMock()
    r.informe    = "# Informe de prueba\n\nContenido simulado."
    r.n_fuentes  = 3
    r.duracion_s = 0.1
    r.ruta_informe = "/tmp/test_informe.md"
    return r


@pytest.fixture
def client():
    """TestClient con mock del agente para tests rápidos."""
    with TestClient(app) as c:
        yield c


# ── Tests rápidos (mock) ──────────────────────────────────────────────────────

def test_health(client):
    r = client.get("/health")
    assert r.status_code == 200
    assert r.json()["status"] == "ok"


def test_crear_investigacion_devuelve_202(client):
    r = client.post("/research", json={"tema": "Inteligencia artificial en la industria agroalimentaria gallega"})
    assert r.status_code == 202
    data = r.json()
    assert "job_id" in data
    assert data["status"] == "pending"


def test_tema_demasiado_corto_da_422(client):
    r = client.post("/research", json={"tema": "corto"})
    assert r.status_code == 422


def test_job_no_existente_da_404(client):
    r = client.get("/jobs/inexistente")
    assert r.status_code == 404


def test_informe_job_pending_da_409(client):
    """Un job en estado pending no tiene informe disponible → 409."""
    r = client.post("/research", json={"tema": "Bioeconomía forestal en Galicia y sus aplicaciones"})
    job_id = r.json()["job_id"]

    # Sin esperar a que termine
    r2 = client.get(f"/jobs/{job_id}/report")
    assert r2.status_code == 409


def test_listar_jobs(client):
    r = client.get("/jobs")
    assert r.status_code == 200
    data = r.json()
    assert "total" in data
    assert "jobs" in data


def test_eliminar_job_existente(client):
    r = client.post("/research", json={"tema": "Tendencias de digitalización en el sector pesquero"})
    job_id = r.json()["job_id"]

    r2 = client.delete(f"/jobs/{job_id}")
    assert r2.status_code == 204

    r3 = client.get(f"/jobs/{job_id}")
    assert r3.status_code == 404


def test_eliminar_job_inexistente_da_404(client):
    r = client.delete("/jobs/no-existe")
    assert r.status_code == 404


def test_informe_job_completado(client):
    """Inyecta un job completado directamente en el store y verifica GET /report."""
    from jobs import Job
    from datetime import datetime

    job = Job(
        id="test99",
        tema="Tema de prueba suficientemente largo",
        status=EstadoJob.COMPLETED,
        informe="# Informe\nContenido de prueba.",
        n_fuentes=2,
        duracion_s=5.0,
        completado_en=datetime.utcnow(),
    )
    store._jobs["test99"] = job

    r = client.get("/jobs/test99/report")
    assert r.status_code == 200
    data = r.json()
    assert "informe" in data
    assert data["n_fuentes"] == 2

    # Limpieza
    store.eliminar("test99")


# ── Test lento (agente real) ──────────────────────────────────────────────────

@pytest.mark.lento
def test_ciclo_completo_con_agente_real(client):
    """
    Test de integración completo: crea un job, hace polling hasta completado
    y verifica que el informe tiene contenido.
    Requiere OPENAI_API_KEY. Puede tardar hasta 2 minutos.
    """
    r = client.post("/research",
        json={"tema": "Aplicaciones de machine learning en la industria conservera gallega"})
    assert r.status_code == 202
    job_id = r.json()["job_id"]

    # Polling hasta completado (máx. 3 minutos)
    inicio = time.time()
    while time.time() - inicio < 180:
        r2 = client.get(f"/jobs/{job_id}")
        data = r2.json()
        print(f"  Status: {data[\'status\']} | {data[\'progreso\']}")
        if data["status"] == "completed":
            break
        if data["status"] == "failed":
            pytest.fail(f"Job fallido: {data[\'error\']}")  
        time.sleep(5)
    else:
        pytest.fail("Timeout: el job no terminó en 3 minutos")

    # Recupera el informe
    r3 = client.get(f"/jobs/{job_id}/report")
    assert r3.status_code == 200
    informe = r3.json()["informe"]
    assert len(informe) > 200, "El informe parece demasiado corto"
    print(f"\nInforme ({len(informe)} chars):\n{informe[:300]}...")
'''

(PROJECT_DIR / "test_api.py").write_text(TEST_API_PY.strip())
print("✓ test_api.py generado")

## Celda 5 — Ejecutar los tests rápidos

In [ ]:
import subprocess

resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "test_api.py", "-v", "-m", "not lento", "--tb=short"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
)
print(resultado.stdout)
if resultado.returncode != 0:
    print(resultado.stderr)

## Celda 6 — Cliente Python para polling

Un helper reutilizable para llamar a la API desde cualquier script o notebook.

In [ ]:
CLIENT_PY = '''
"""Cliente Python para la ResearchAgent API."""
import time
import httpx


class ResearchClient:
    """
    Cliente que encapsula el ciclo fire-and-poll.

    Uso:
        c = ResearchClient("http://localhost:8000")
        informe = c.investigar_y_esperar("IA en acuicultura", timeout=120)
        print(informe)
    """

    def __init__(self, base_url: str = "http://localhost:8000"):
        self.base = base_url.rstrip("/")
        self.http = httpx.Client(timeout=30)

    def iniciar(self, tema: str) -> str:
        """Inicia una investigación y devuelve el job_id."""
        r = self.http.post(f"{self.base}/research", json={"tema": tema})
        r.raise_for_status()
        return r.json()["job_id"]

    def estado(self, job_id: str) -> dict:
        r = self.http.get(f"{self.base}/jobs/{job_id}")
        r.raise_for_status()
        return r.json()

    def informe(self, job_id: str) -> str:
        r = self.http.get(f"{self.base}/jobs/{job_id}/report")
        r.raise_for_status()
        return r.json()["informe"]

    def investigar_y_esperar(
        self,
        tema: str,
        timeout: int = 180,
        intervalo: float = 5.0,
        verbose: bool = True,
    ) -> str:
        """
        Inicia una investigación y hace polling hasta que termine.
        Devuelve el informe como string Markdown.
        """
        job_id = self.iniciar(tema)
        if verbose:
            print(f"Job iniciado: {job_id}")

        inicio = time.time()
        while time.time() - inicio < timeout:
            est = self.estado(job_id)
            if verbose:
                elapsed = int(time.time() - inicio)
                print(f"  [{elapsed}s] {est[\'status\']} — {est[\'progreso\']}")

            if est["status"] == "completed":
                return self.informe(job_id)
            if est["status"] == "failed":
                raise RuntimeError(f"Investigación fallida: {est[\'error\']}")

            time.sleep(intervalo)

        raise TimeoutError(f"Job {job_id} no terminó en {timeout}s")
'''

(PROJECT_DIR / "client.py").write_text(CLIENT_PY.strip())
print("✓ client.py generado")
print()
print("Uso del cliente (con la API corriendo):")
print("""
  from client import ResearchClient

  c = ResearchClient("http://localhost:8000")
  informe = c.investigar_y_esperar("IA en la cadena de valor del vino gallego")
  print(informe)
""")

## Celda 7 — Arrancar la API en local (opcional)

Ejecuta la celda, luego abre http://localhost:8000/docs en el navegador
para ver la documentación Swagger y probar los endpoints manualmente.

**Detén el kernel para parar el servidor.**

In [ ]:
# Descomenta para arrancar el servidor (bloquea el kernel)
# import subprocess, sys
# subprocess.run(
#     [sys.executable, "-m", "uvicorn", "api:app", "--reload", "--port", "8000"],
#     cwd=str(PROJECT_DIR),
# )

print("Para arrancar la API:")
print(f"  cd {PROJECT_DIR}")
print("  uvicorn api:app --reload --port 8000")
print()
print("Docs: http://localhost:8000/docs")
print("Prueba rápida:")
print('  curl -X POST http://localhost:8000/research \\')
print('       -H "Content-Type: application/json" \\')
print('       -d \'{"tema": "Aplicaciones de IA en la pesca gallega"}\'\'')

## Reto — Añadir progreso detallado durante la ejecución

El campo `progreso` ahora solo dice `"Planificando..."` o `"Completado"`. Mejóralo para que
muestre qué subtarea está ejecutando en tiempo real: `"2/4 subtareas completadas"`.

El reto tiene dos partes:

**Parte A — Callback en ResearchAgent**

Modifica `ResearchAgent.investigar()` en `agent/agent.py` para aceptar un callback
opcional que se llama después de cada subtarea:

```python
def investigar(
    self,
    tema: str,
    on_progreso: Callable[[int, int, str], None] | None = None,
) -> ResultadoInvestigacion:
    ...
    for i, subtarea in enumerate(plan, 1):
        resultado = ejecutar_subtarea(subtarea)
        resultados.append(resultado)
        if on_progreso:
            on_progreso(i, len(plan), subtarea["descripcion"])
```

**Parte B — Actualizar el store desde el callback**

En `api.py`, pasa el callback a `investigar()` para que actualice el campo `progreso`:

```python
def _on_progreso(completadas: int, total: int, desc: str):
    store.actualizar(
        job_id,
        progreso=f"{completadas}/{total} subtareas — {desc[:40]}"
    )

resultado = await asyncio.to_thread(agente.investigar, tema, _on_progreso)
```

Verifica que funciona haciendo polling con `GET /jobs/{job_id}` mientras corre
el test lento y comprobando que `progreso` cambia entre llamadas.

## Resumen de la semana 16

| Concepto | Dónde lo aplicaste |
|---------|-------------------|
| Patrón fire-and-poll | `api.py` + `POST /research` |
| HTTP 202 Accepted | `POST /research` |
| HTTP 409 Conflict | `GET /jobs/{id}/report` cuando no completado |
| Máquina de estados (pending→running→completed/failed) | `jobs.py` + `EstadoJob` |
| `asyncio.create_task` para background | `api.py` |
| `asyncio.to_thread` para no bloquear | `_ejecutar_investigacion()` |
| Tests con mock del agente | `test_api.py` (8 tests rápidos) |
| Test de integración con agente real | `test_api.py` (`@pytest.mark.lento`) |
| Cliente con polling y backoff | `client.py` |
| Progreso en tiempo real | Reto |

**Bloque 7 continúa.** La semana 17 añade LLMOps: observabilidad, trazas y métricas de coste
para el agente en producción.